# Deep-Sea Classification: Context Reliance & Depth Degradation
This notebook imports from the project's `.py` modules and runs the full pipeline on Colab's GPU.

## 0 · Setup: Clone repo & install dependencies

In [ ]:
# Run once per session — clone the repo and cd into it
!git clone https://github.com/<your-org>/deepsea-crs.git
%cd deepsea-crs

# If already cloned, just pull the latest
# %cd /content/deepsea-crs
# !git pull origin main

In [ ]:
# Install any extra packages not on Colab by default
# !pip install fathomnet grad-cam  # adjust as needed

In [ ]:
# Verify GPU is available
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Import project modules
import config
import data
import model
import attribution
import evaluation
import analysis

## 1 · Data loading & preprocessing

In [ ]:
# Load FathomNet metadata and inspect depth distribution
metadata = data.load_fathomnet_metadata()
print(f'Total images: {len(metadata)}')
print(f'Depth range: {metadata["depth_m"].min():.0f}m – {metadata["depth_m"].max():.0f}m')

In [ ]:
# Split into shallow / deep
shallow_df, deep_df = data.split_shallow_deep(metadata, config.DEPTH_SHALLOW_MAX)
print(f'Shallow: {len(shallow_df)} | Deep: {len(deep_df)}')

In [ ]:
# Build DataLoaders
train_loader = data.get_dataloaders(shallow_df, split='train')
val_loader   = data.get_dataloaders(shallow_df, split='val')
shallow_test_loader = data.get_dataloaders(shallow_df, split='test')
deep_test_loader    = data.get_dataloaders(deep_df, split='test')

## 2 · Train classifier (shallow-depth only)

In [ ]:
species_list = sorted(shallow_df['species'].unique())
classifier = model.build_classifier(num_classes=len(species_list))
classifier, history = model.train(classifier, train_loader, val_loader)
model.save_checkpoint(classifier, config.MODEL_SAVE_PATH + 'shallow_resnet50.pt')

## 3 · Attribution & Context Reliance Score

In [ ]:
# Compute per-species CRS on shallow test set
crs_scores = attribution.compute_species_crs(classifier, shallow_test_loader, species_list)
print('CRS per species:', crs_scores)

In [ ]:
# Baseline attribution maps
uniform_crs  = {}  # TODO: compute CRS with uniform baseline per species
gaussian_crs = {}  # TODO: compute CRS with Gaussian baseline per species

## 4 · Evaluation & degradation

In [ ]:
# Per-species accuracy on shallow and deep
shallow_acc = evaluation.per_species_accuracy(classifier, shallow_test_loader, species_list)
deep_acc    = evaluation.per_species_accuracy(classifier, deep_test_loader, species_list)
degradation = evaluation.compute_degradation(shallow_acc, deep_acc)

# Majority-class baseline
majority_acc = evaluation.majority_class_baseline(shallow_test_loader)
print(f'Majority-class baseline: {majority_acc:.3f}')

In [ ]:
# Control model (trained on all depths)
control = model.build_control_model(num_classes=len(species_list))
all_loader = data.get_dataloaders(metadata, split='train')
control, _ = model.train(control, all_loader, val_loader)
control_results = evaluation.evaluate_control_model(control, shallow_test_loader, deep_test_loader, species_list)

## 5 · Analysis: CRS vs degradation

In [ ]:
# Core hypothesis test
rho, p_value = analysis.spearman_correlation(crs_scores, degradation)
print(f'Spearman ρ = {rho:.3f}, p = {p_value:.4f}')

In [ ]:
# Plots
analysis.plot_crs_vs_degradation(crs_scores, degradation, save_path='results/crs_vs_degradation.png')
analysis.plot_baseline_comparison(crs_scores, uniform_crs, gaussian_crs, save_path='results/baseline_crs.png')
analysis.plot_control_comparison(degradation, control_results, save_path='results/control_comparison.png')

In [ ]:
# Summary table
summary = analysis.generate_summary_table(crs_scores, degradation, rho, p_value)
summary

## 6 · Push results back to GitHub

In [ ]:
# !git add results/
# !git commit -m "Add experiment results"
# !git push origin main